# 01 — Historical ETF data

This notebook downloads daily adjusted closing prices for the 12 ETFs
used in the market risk study. It saves a data snapshot and records
the source, requested date range, and download date.

The acquisition ends before 2018. The evaluation period is reserved
until the model is frozen.

In [1]:
%pip install -q yfinance pyarrow

In [2]:
import json
from pathlib import Path

import pandas as pd
import pyarrow
import yfinance as yf

print("pandas:", pd.__version__)
print("yfinance:", yf.__version__)
print("pyarrow:", pyarrow.__version__)

pandas: 2.2.3
yfinance: 0.2.66
pyarrow: 23.0.1


In [3]:
START_DATE = "2008-01-01"
END_DATE = "2018-01-01"

def download_daily_data(tickers):
    return yf.download(
        tickers=tickers,
        start=START_DATE,
        end=END_DATE,
        interval="1d",
        auto_adjust=False,
        group_by="column",
        multi_level_index=True,
        ignore_tz=True,
        keepna=True,
        progress=False,
    )

pilot_tickers = ["SPY", "IEF"]
pilot_data = download_daily_data(pilot_tickers)

display(pilot_data.head())

ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['SPY']: OperationalError('database is locked')


Price       Adj Close          Close           High            Low      \
Ticker            IEF SPY        IEF SPY        IEF SPY        IEF SPY   
Date                                                                     
2008-01-02  55.000565 NaN  87.629997 NaN  87.730003 NaN  86.940002 NaN   
2008-01-03  55.113503 NaN  87.809998 NaN  87.839996 NaN  87.309998 NaN   
2008-01-04  55.257904 NaN  88.040001 NaN  88.239998 NaN  87.949997 NaN   
2008-01-07  55.358276 NaN  88.199997 NaN  88.300003 NaN  87.870003 NaN   
2008-01-08  55.477562 NaN  88.389999 NaN  88.419998 NaN  87.870003 NaN   

Price            Open      Volume      
Ticker            IEF SPY     IEF SPY  
Date                                   
2008-01-02  86.989998 NaN  564700 NaN  
2008-01-03  87.500000 NaN  198600 NaN  
2008-01-04  88.000000 NaN  368700 NaN  
2008-01-07  87.870003 NaN  341100 NaN  
2008-01-08  88.089996 NaN  293200 NaN

In [6]:
def select_adjusted_close(response, tickers):
    if response is None or response.empty:
        raise ValueError("Il download non ha restituito dati.")

    if "Adj Close" not in response.columns.get_level_values(0):
        raise ValueError("Il campo Adj Close manca nella risposta.")

    adjusted = response["Adj Close"]

    missing_tickers = [
        ticker for ticker in tickers
        if ticker not in adjusted.columns or adjusted[ticker].isna().all()
    ]

    if missing_tickers:
        raise ValueError(f"Nessun prezzo disponibile per: {missing_tickers}")

    return adjusted.loc[:, tickers].copy()

pilot_prices = select_adjusted_close(pilot_data, pilot_tickers)

display(pilot_prices.head())
print("Righe e colonne:", pilot_prices.shape)

ValueError: Nessun prezzo disponibile per: ['SPY']

In [5]:
TICKERS = [
    "SPY", "QQQ", "IWM", "EFA", "EEM", "VNQ",
    "TLT", "IEF", "LQD", "HYG", "GLD", "DBC",
]

market_data = download_daily_data(TICKERS)
downloaded_at = pd.Timestamp.now(tz="UTC").isoformat()

prices = select_adjusted_close(market_data, TICKERS)

if prices.index.min() < pd.Timestamp(START_DATE):
    raise ValueError("Il file contiene date precedenti all'inizio richiesto.")

if prices.index.max() >= pd.Timestamp(END_DATE):
    raise ValueError("Il file contiene date del periodo di valutazione.")

print("Numero di righe:", len(prices))
print("Numero di ETF:", prices.shape[1])
print("Prima data:", prices.index.min().date())
print("Ultima data:", prices.index.max().date())
print("Ticker:", prices.columns.tolist())

display(prices.head())

Numero di righe: 2518
Numero di ETF: 12
Prima data: 2008-01-02
Ultima data: 2017-12-29
Ticker: ['SPY', 'QQQ', 'IWM', 'EFA', 'EEM', 'VNQ', 'TLT', 'IEF', 'LQD', 'HYG', 'GLD', 'DBC']


Ticker,SPY,QQQ,IWM,EFA,EEM,VNQ,TLT,IEF,LQD,HYG,GLD,DBC
Date,,,,,,,,,,,,
2008-01-02,102.973946,43.264091,58.372086,44.838005,33.408066,28.255737,52.945423,55.000565,51.003300,31.048357,84.860001,26.991880
2008-01-03,102.924202,43.444317,57.727505,44.912518,33.708637,27.374914,52.872509,55.113503,50.835007,30.989080,85.570000,27.199957
2008-01-04,100.401901,41.539017,55.987831,43.892159,32.696136,26.263496,52.883717,55.257904,51.253307,30.954746,85.129997,27.041822
2008-01-07,100.316666,41.341625,56.112091,43.938015,32.935715,26.507910,53.113667,55.358276,51.190773,30.770628,84.769997,26.550756
2008-01-08,98.696739,40.268837,54.380177,43.691528,32.680325,25.590191,53.052025,55.477562,51.094620,30.592787,86.779999,26.792126


In [7]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [8]:
raw_dir = Path("/content/drive/MyDrive/quant-research-lab/data/raw")
raw_dir.mkdir(parents=True, exist_ok=True)

prices_path = raw_dir / "etf_adj_close_2008_2017.parquet"
metadata_path = raw_dir / "etf_adj_close_2008_2017_metadata.json"

prices.to_parquet(prices_path, engine="pyarrow", index=True)

metadata = {
    "source": "Yahoo Finance via yfinance",
    "price_field": "Adj Close",
    "interval": "1d",
    "auto_adjust": False,
    "start_inclusive": START_DATE,
    "end_exclusive": END_DATE,
    "tickers": TICKERS,
    "downloaded_at_utc": downloaded_at,
    "rows": len(prices),
    "first_observation": str(prices.index.min().date()),
    "last_observation": str(prices.index.max().date()),
    "yfinance_version": yf.__version__,
    "pandas_version": pd.__version__,
    "pyarrow_version": pyarrow.__version__,
}

metadata_path.write_text(
    json.dumps(metadata, indent=2),
    encoding="utf-8",
)

print("Prezzi salvati in", prices_path)
print("Metadati salvati in", metadata_path)

Prezzi salvati in /content/drive/MyDrive/quant-research-lab/data/raw/etf_adj_close_2008_2017.parquet
Metadati salvati in /content/drive/MyDrive/quant-research-lab/data/raw/etf_adj_close_2008_2017_metadata.json


In [9]:
reloaded_prices = pd.read_parquet(prices_path, engine="pyarrow")

pd.testing.assert_frame_equal(reloaded_prices, prices, check_exact=True)

print("Verifica superata: il file riaperto coincide con la tabella salvata.")
print("Dimensioni:", reloaded_prices.shape)

Verifica superata: il file riaperto coincide con la tabella salvata.
Dimensioni: (2518, 12)
